# Enveda CASMI 2026 — final inference

Run after attaching four Kaggle Inputs: the competition data, the versioned source-code dataset, the October 2026 COCONUT CSV-lite ZIP, and the private ranker-training-features dataset. Keep internet disabled. This notebook creates `submission.csv` in `/kaggle/working`.

In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/casmi26')
WORK.mkdir(parents=True, exist_ok=True)

def one(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise FileNotFoundError(f'Expected one {label}; found {len(paths)}: {paths[:8]}')
    return paths[0]

TRAIN = one(INPUT.rglob('train.parquet'), 'competition train.parquet')
TEST = one(INPUT.rglob('test.parquet'), 'competition test.parquet')
SOURCE_ROOT = one((p.parent for p in INPUT.rglob('retrieve.py') if (p.parent / 'prepare_candidates.py').exists()), 'source-code dataset')
COCONUT_ZIP = one((p for p in INPUT.rglob('*.zip') if 'coconut' in p.name.lower()), 'COCONUT CSV-lite ZIP')
RANKER_FEATURES = one(INPUT.rglob('ranker_training_features.parquet'), 'ranker_training_features.parquet')
sys.path.insert(0, str(SOURCE_ROOT))
print('Train:', TRAIN)
print('Test:', TEST)
print('Source:', SOURCE_ROOT)
print('COCONUT:', COCONUT_ZIP)
print('Ranker features:', RANKER_FEATURES)
commit_file = SOURCE_ROOT / 'SOURCE_COMMIT.txt'
print('Source commit:', commit_file.read_text().strip() if commit_file.exists() else 'missing')

In [ ]:
import rdkit
parts = tuple(int(part) for part in rdkit.__version__.split('.')[:3])
if parts != (2026, 3, 3):
    raise RuntimeError(f'Competition-pinned RDKit 2026.03.3 required; found {rdkit.__version__}. Attach the pinned RDKit Kaggle Dataset as an Input.')
print('RDKit:', rdkit.__version__)

## Fit the CPU reranker

The attached feature table is derived from the training set with molecule-grouped validation. The final model is fit on all eligible training groups. This step uses CPU; the GPU is not needed for the current model.

In [ ]:
MODEL = WORK / 'final_reranker.joblib'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'fit_reranker.py'),
    '--features', str(RANKER_FEATURES), '--model-output', str(MODEL),
], check=True)

## Generate structure-database candidates

Filter the attached COCONUT snapshot around each hidden-test precursor mass, then score candidate fragments and apply the learned reranker.

In [ ]:
CANDIDATES = WORK / 'coconut_candidates.parquet'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'prepare_candidates.py'),
    '--input', str(COCONUT_ZIP), '--output', str(CANDIDATES),
    '--query-parquet', str(TEST), '--window-ppm', '50',
    '--source-version', '2026-10', '--workers', '4',
], check=True)

DATABASE_FEATURES = WORK / 'database_features.parquet'
DATABASE_FRAGMENT_PREDICTIONS = WORK / 'database_fragment_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'retrieve_fragments.py'),
    '--candidates', str(CANDIDATES), '--queries', str(TEST),
    '--window-ppm', '10', '--features-output', str(DATABASE_FEATURES),
    '--output', str(DATABASE_FRAGMENT_PREDICTIONS),
], check=True)

DATABASE_PREDICTIONS = WORK / 'database_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'predict_reranker.py'),
    '--model', str(MODEL), '--features', str(DATABASE_FEATURES),
    '--output', str(DATABASE_PREDICTIONS),
], check=True)

## Retrieve library spectra and write the blended submission

The spectral route stays first. Database predictions only fill unused positions, so they cannot push a spectral candidate down the ranking.

In [ ]:
SPECTRAL_PREDICTIONS = WORK / 'spectral_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'retrieve.py'),
    '--train', str(TRAIN), '--test', str(TEST),
    '--output', str(SPECTRAL_PREDICTIONS), '--top-k', '25',
], check=True)

BLENDED = WORK / 'blended_predictions.csv'
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'blend_predictions.py'),
    '--primary', str(SPECTRAL_PREDICTIONS),
    '--fallback', str(DATABASE_PREDICTIONS), '--test', str(TEST),
    '--output', str(BLENDED),
], check=True)

SUBMISSION = Path('/kaggle/working/submission.csv')
subprocess.run([
    sys.executable, str(SOURCE_ROOT / 'write_submission.py'),
    '--predictions', str(BLENDED), '--test', str(TEST),
    '--output', str(SUBMISSION),
], check=True)
print('Submission ready:', SUBMISSION, 'bytes:', SUBMISSION.stat().st_size)